# Day 26: Model Robustness Testing

YBI Foundation — Python AI & Data Science Internship

**Project:** AI-Based Student Performance Prediction System

## Objective

Measure prediction sensitivity while checking batch and column-order invariance.

## Theory

A model should produce the same result regardless of batch composition and DataFrame column ordering. Plausible input variations may change predictions; this is a sensitivity check, not a causal intervention.

## Prerequisites

Install `requirements-dev.txt` (and `requirements-api.txt` for API work) in the active Python environment. Use the repository checkout and the saved Day 17 model. The setup below locates or clones the repository for Colab. Run cells in order.


In [1]:
from pathlib import Path
import os, sys, subprocess, json
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Project:", root.name)


Project: IYB_INTERNSHIP


Load the saved model and reference records.

In [2]:
from src.explainability import load_explainability_pipeline,load_reference_dataset
import numpy as np
import pandas as pd
pipeline=load_explainability_pipeline()
X,y=load_reference_dataset()
baseline=pipeline.predict(X)

Verify invariance across order and batching.

In [3]:
assert np.array_equal(baseline,pipeline.predict(X[X.columns[::-1]]))
individual=np.concatenate([pipeline.predict(X.iloc[[i]]) for i in range(len(X))])
assert np.array_equal(baseline,individual)
assert np.allclose(pipeline.predict_proba(X).sum(axis=1),1)
print("Column order, single/batch parity and probability normalization: PASS")

Column order, single/batch parity and probability normalization: PASS


Measure sensitivity within documented integer bounds.

In [4]:
records=[]
for feature,upper in [("absences",93),("studytime",4),("failures",3)]:
    changed=X.copy()
    changed[feature]=(changed[feature]+1).clip(upper=upper)
    changed_count=int((changed[feature]!=X[feature]).sum())
    pred=pipeline.predict(changed)
    records.append({"feature":feature,"changed_inputs":changed_count,"changed_predictions":int((pred!=baseline).sum()),"fraction_of_all_rows":float((pred!=baseline).mean())})
report=pd.DataFrame(records)
report.to_csv("reports/robustness.csv",index=False)
print(report.to_string(index=False))

  feature  changed_inputs  changed_predictions  fraction_of_all_rows
 absences             130                    4              0.030769
studytime             124                   12              0.092308
 failures             128                   63              0.484615


## Findings and interpretation

Ordering and batch assertions enforce serving consistency. The sensitivity table records actual changes and can highlight brittle behavior for future review.

## Limitations

Modified records are explicit test scenarios, not new students or educational outcome evidence. No robustness score is used to choose or retrain the model.

## Development log

Implemented and ran the cells above during the repair and extension session on 2 November 2026. The saved outputs are produced by the kernel.

## Conclusion

Measure prediction sensitivity while checking batch and column-order invariance.

## Next-day plan

Centralize input validation and reliable prediction handling.